# Pós-Experimento em Modo Somente-Leitura: LaTeX, SVG e Conferências

Este notebook demonstra o desacoplamento entre execução (Etapa 4) e publicação (Etapa 6):
- Carregamos um experimento já concluído via `load_experiment_artifacts_readonly`;
- Geramos tabelas `LaTeX booktabs` e gráficos `SVG` em Python/NumPy puro;
- Exportamos um pacote `reduced_no_raw_data` com checksums SHA-256 relativos.

In [ ]:
from pathlib import Path
import tempfile
import numpy as np

from experimental_ml.core.enums import MetricDirection, TaskType
from experimental_ml.data import (
    DatasetSchemaSpec,
    create_preparation_plan,
    load_dataset,
    profile_dataset,
    sanitize_dataset,
)
from experimental_ml.features import FeatureScenario, ScenarioMode
from experimental_ml.models import (
    DummyBaselineModel,
    ModelCapabilities,
    ModelFamily,
    ModelSpec,
)
from experimental_ml.orchestration import load_cell_artifacts, run_experiment_matrix
from experimental_ml.reporting import (
    PackageMode,
    ReportAudience,
    ReportContentSelection,
    ReportFamily,
    build_figure_spec,
    build_reproducibility_package,
    build_table_model,
    load_experiment_artifacts_readonly,
    render_figure,
    render_table_latex,
    select_report_content,
)
from experimental_ml.statistics import (
    ComparisonScope,
    ComparisonSpec,
    consolidate_descriptive,
    evaluate_method_suitability,
    run_statistical_comparisons,
    save_statistics_artifacts,
    synthesize_conclusions,
)
from experimental_ml.validation import (
    EvaluationRegime,
    PartitionConfig,
    PartitionStrategy,
    build_partition_plan,
)

## 1. Preparação de um Experimento Concluído (Etapas 1 a 5)

In [ ]:
exp_dir = Path(tempfile.mkdtemp(prefix='expml_nb04_'))
rng = np.random.default_rng(123)
dados = {
    'x1': rng.normal(0.0, 1.0, size=30).tolist(),
    'x2': rng.normal(1.0, 0.5, size=30).tolist(),
    'alvo': [0 if i < 15 else 1 for i in range(30)],
}
bundle = load_dataset(dados, DatasetSchemaSpec(target_col='alvo', task_type=TaskType.BINARY_CLASSIFICATION))
profile = profile_dataset(bundle)
view = sanitize_dataset(bundle, profile, create_preparation_plan(profile, rules=[]))
cen = FeatureScenario(
    scenario_id='todas', name='Todas', description='Todas',
    scientific_justification='Base', mode=ScenarioMode.FIXED, selected_features=('x1', 'x2'),
)
m1 = ModelSpec(
    model_id='dummy_freq', display_name='Most Frequent', family=ModelFamily.BASELINE,
    estimator_factory=lambda seed=None: DummyBaselineModel(strategy='most_frequent'),
    capabilities=ModelCapabilities(supports_predict_proba=True),
    description='Dummy', scientific_justification='Baseline',
)
part_plan = build_partition_plan(
    view,
    PartitionConfig(strategy=PartitionStrategy.STRATIFIED_KFOLD, regime=EvaluationRegime.CV_ONLY, n_splits=3, master_seed=42),
)
state = run_experiment_matrix(
    model_specs=[m1], scenarios=[cen], partition_plan=part_plan,
    sanitized_view=view, run_dir=exp_dir / 'runs', metrics=('accuracy',), master_seed=42, overwrite=True,
)
cells = [load_cell_artifacts(cid, exp_dir / 'runs')[0] for cid in state.cell_statuses]
cons = consolidate_descriptive(cells, total_folds_per_config=len(part_plan.folds))
suit = evaluate_method_suitability(part_plan, cons)
comp_spec = ComparisonSpec(
    metrics_of_interest=('accuracy',),
    metric_directions={'accuracy': MetricDirection.MAXIMIZE},
    comparison_scope=ComparisonScope.ALL_PAIRWISE,
    alpha=0.05,
)
comp = run_statistical_comparisons(cons, part_plan, comp_spec, suit)
conc = synthesize_conclusions(cons, suit, comp, primary_metric='accuracy', primary_direction=MetricDirection.MAXIMIZE)
save_statistics_artifacts(exp_dir / 'statistics', cons, suit, comp, conc, force=True)
print('Experimento salvo em:', exp_dir)

## 2. Leitura Somente-Leitura, Tabela LaTeX, Figura SVG e Pacote Reduzido

In [ ]:
loaded = load_experiment_artifacts_readonly(exp_dir)
selection = select_report_content(
    loaded,
    ReportContentSelection(
        audience=ReportAudience.ARTICLE_COMPACT,
        families=(ReportFamily.D_PREDICTIVE_PERFORMANCE,),
    ),
)

tabela = build_table_model(ReportFamily.D_PREDICTIVE_PERFORMANCE, loaded, selection)
latex_str = render_table_latex(tabela)
assert '\\toprule' in latex_str
print('Prévia LaTeX:\n', latex_str[:220])

fig_spec = build_figure_spec(ReportFamily.D_PREDICTIVE_PERFORMANCE, loaded, selection)
svg_bytes = render_figure(fig_spec, fmt='svg')
try:
    from IPython.display import SVG, display
    display(SVG(svg_bytes))
except ImportError:
    print(f'SVG gerado com {len(svg_bytes)} bytes.')

pacote_dir = exp_dir / 'pacote_conferencia'
manifest = build_reproducibility_package(
    loaded,
    output_dir=pacote_dir,
    mode=PackageMode.REDUCED_NO_RAW_DATA,
    source_experiment_dir=exp_dir,
    force=True,
)
assert manifest.reproduction_level == 'REDUCED_NO_RAW_DATA'
assert (pacote_dir / 'reproducibility' / 'checksums.sha256').is_file()
print('Pacote exportado com sucesso:', manifest.reproduction_level)